In [ ]:
import os
import mne
import numpy as np
import warnings
from typing import List, Dict, Tuple
import import_ipynb
import eeg_preprocessing_interface as eeg_pp
import eeg_band_separation as eeg_band
import eeg_complexity_feture as eeg_feture
from tqdm import tqdm

# Set MNE logging level to ERROR to suppress unnecessary logs
mne.set_log_level("ERROR")

# Ignore RuntimeWarning about non-standard MNE filename conventions
warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
    message="This filename.*does not conform to MNE naming conventions"
)

# Ignore UserWarning about legacy pick_channels() function usage
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message="NOTE: pick_channels\\(\\) is a legacy function. New code should use inst.pick\\(\\.\\.\\.\\)."
)

In [ ]:
# 1. Basic Configuration (Subjects, Directories, Time Points, etc.)
sub_10hz = ["100306", "100412", "100515", "100723", "100927", "101029", "101139",#"100102", "100203" "100617"
            "101449", "101551", "101656", "101758", "101861", "101965", "102274", "102375", "102478", "102580"]

sub_sham = ["300207", "300308", "300409", "300618", "300720","300822", "300925", "301041", "301250", #"300104",
            "301354", "301455", "301560", "301662", "301766", "301867", "302177", "302282", "302384"]  # Removed "300513"

# Directory Configuration
base_dir_10hz = r'D:\山东第一医科大学\数据\TI_TASK_DATA\Task2\八因子\10HZ'
base_dir_sham = r'D:\山东第一医科大学\数据\TI_TASK_DATA\Task2\八因子\SHAM'
out_base_dir = r'D:\山东第一医科大学\数据\TI_TASK_DATA\Task2\八因子\预处理'  # Can be used to save results if needed
#target_bands = ['delta', 'theta', 'alpha', 'lbeta', 'hbeta', 'gamma1', 'gamma2', 'gamma3', 'gamma4', 'gamma5', 'gamma6']
# Experimental Dimension Configuration
time_points = ['pre','post'] 
emotions = ['happy','sad']
groups = {
    #'10hz': {'subjects': sub_10hz, 'source_dir': base_dir_10hz},
    'sham': {'subjects': sub_sham, 'source_dir': base_dir_sham}
}


# Initialize Interfaces
preprocessor = eeg_pp.EEGPreprocessor()
#band_sep = eeg_band.EEGBandSeparator()

In [ ]:
i = 0  # Directory index counter

# Add progress bar for group loop
for group_name, group_info in tqdm(groups.items(), desc="Processing Groups", unit="group"):
    source_dir = group_info['source_dir']
    subject_list = group_info['subjects']  # 20 subjects
    
    # Add progress bar for time point loop
    for time in tqdm(time_points, desc=f"Processing {group_name} Time Points", unit="time point", leave=False):
        # Add progress bar for emotion loop
        for emotion in tqdm(emotions, desc=f"Processing {time} Emotions", unit="emotion", leave=False):
            
            # 3. Iterate over subjects with progress bar
            for sub_idx, sub in enumerate(tqdm(subject_list, desc="Processing Subjects", unit="subject", leave=False)):
                sub_source_dir = os.path.join(source_dir, time, sub, emotion)
                i += 1
                print(f"\n=== Directory Index: {i} ===")
                print(f"Current Directory: {sub_source_dir}")

                sub_save_dir = os.path.join(out_base_dir, group_name, time, emotion, sub)
                os.makedirs(sub_save_dir, exist_ok=True)
                
                # Iterate over factors
                for factor_num in tqdm(range(1, 9), desc="Processing Factors", unit="factor", leave=False):
                    factor_idx = factor_num - 1
                    factor_filename = f"factor_{factor_num}.fif"
                    factor_path = os.path.join(sub_source_dir, factor_filename)
                    # Load + Preprocess + Band separation + Feature calculation
                    raw = mne.io.read_raw_fif(factor_path, preload=True, verbose=False)
                    raw_clean, ica = preprocessor.preprocess(raw)
                    #band_data = band_sep.separate_bands(raw=raw_clean)
                    out_base = os.path.join(sub_save_dir, factor_filename)
                    # 7. Save FIF file (filename uses factor_X.fif directly, path contains all hierarchical info)
                    raw_clean.save(out_base, overwrite=True, verbose=False)

                    print(f"✅ FIF file saved: {factor_path}")